# Question 2: Road Traffic Accident Severity

## Stage 4: Data Cleaning and Preprocessing

Group: Group 2

Members: Armand Ikirezi and Mariam Nyirarukundo

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(r"D:\MYProject\ML_Element1")

raw_path = (
    project_dir
    / "q2_own_scenario"
    / "data"
    / "raw"
    / "rwanda_traffic_accidents_synthetic.csv"
)

df = pd.read_csv(raw_path)

print("Original shape:", df.shape)


Original shape: (1608, 13)


In [2]:

df_clean = df.copy()

before = len(df_clean)

df_clean = df_clean.drop_duplicates()

after = len(df_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Removed:", before - after)


Rows before: 1608
Rows after: 1600
Removed: 8


In [3]:

df_clean["district"] = (
    df_clean["district"]
    .astype("string")
    .str.strip()
)

df_clean["road_type"] = (
    df_clean["road_type"]
    .astype("string")
    .str.strip()
    .replace({
        "High way": "Highway"
    })
)

print("District values:")
print(df_clean["district"].unique())

print("\nRoad type values:")
print(df_clean["road_type"].unique())


District values:
<StringArray>
[    'Rusizi',    'Karongi',     'Rubavu',   'Kicukiro',    'Musanze',
  'Nyagatare',       'Huye',    'Muhanga',   'Bugesera',  'Rwamagana',
     'Gasabo', 'Nyarugenge']
Length: 12, dtype: string

Road type values:
<StringArray>
['Market road', 'Highway', 'Urban', 'Rural']
Length: 4, dtype: string


In [4]:

print("Invalid speed limits:")
display(
    df_clean[
        (df_clean["speed_limit_kmh"] <= 0) |
        (df_clean["speed_limit_kmh"] > 130)
    ]
)

print("Invalid response times:")
display(
    df_clean[
        df_clean["response_time_min"] <= 0
    ]
)


Invalid speed limits:


,accident_id,district,road_type,weather,road_surface,lighting,time_of_day,vehicle_type,speed_limit_kmh,vehicles_involved,road_users_affected,response_time_min,severity
521,RWACC-00522,Muhanga,Urban,Clear,Gravel,Daylight,Afternoon,Motorcycle,220,1,5,36.3,Minor
731,RWACC-00732,Gasabo,Urban,Clear,Wet,Daylight,Morning,Truck,180,1,4,41.9,Minor
1237,RWACC-01238,Rubavu,Rural,Rain,Dry,Street lights,Evening,Bus,250,1,3,2.5,Serious
1540,RWACC-01541,Huye,Urban,Clear,Dry,Daylight,Afternoon,Truck,0,2,5,42.6,Serious


Invalid response times:


,accident_id,district,road_type,weather,road_surface,lighting,time_of_day,vehicle_type,speed_limit_kmh,vehicles_involved,road_users_affected,response_time_min,severity
462,RWACC-00463,Rwamagana,Rural,Rain,Dry,Daylight,Morning,Motorcycle,50,2,6,-5.0,Serious


In [5]:

df_clean.loc[
    (df_clean["speed_limit_kmh"] <= 0) |
    (df_clean["speed_limit_kmh"] > 130),
    "speed_limit_kmh"
] = np.nan

df_clean.loc[
    df_clean["response_time_min"] <= 0,
    "response_time_min"
] = np.nan

print("Invalid values corrected.")


Invalid values corrected.


In [6]:

interim_dir = (
    project_dir
    / "q2_own_scenario"
    / "data"
    / "interim"
)

interim_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = interim_dir / "q2_cleaned.csv"

df_clean.to_csv(cleaned_path, index=False)

print("Saved:", cleaned_path)
print("Shape:", df_clean.shape)


Saved: D:\MYProject\ML_Element1\q2_own_scenario\data\interim\q2_cleaned.csv
Shape: (1600, 13)


In [7]:

from sklearn.model_selection import train_test_split

target = "severity"

X = df_clean.drop(
    columns=["severity", "accident_id"]
)

y = df_clean[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)


Training: (1120, 11)
Validation: (240, 11)
Testing: (240, 11)


In [8]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


In [9]:

X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(
    X_validation
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Training processed:", X_train_processed.shape)
print("Validation processed:", X_validation_processed.shape)
print("Test processed:", X_test_processed.shape)


Training processed: (1120, 41)
Validation processed: (240, 41)
Test processed: (240, 41)


In [10]:

from pathlib import Path

processed_dir = (
    project_dir
    / "q2_own_scenario"
    / "data"
    / "processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

feature_names = preprocessor.get_feature_names_out()

X_train_final = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_validation_final = pd.DataFrame(
    X_validation_processed,
    columns=feature_names,
    index=X_validation.index
)

X_test_final = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

train_final = X_train_final.copy()
train_final["severity"] = y_train

validation_final = X_validation_final.copy()
validation_final["severity"] = y_validation

test_final = X_test_final.copy()
test_final["severity"] = y_test

train_final.to_csv(
    processed_dir / "q2_train_final.csv",
    index=False
)

validation_final.to_csv(
    processed_dir / "q2_validation_final.csv",
    index=False
)

test_final.to_csv(
    processed_dir / "q2_test_final.csv",
    index=False
)

print("Final datasets saved successfully.")


Final datasets saved successfully.
